# RONA

> Risk of non-adaptedness

The Risk of non-adaptedness (RONA) was first developed by @Rellstab_2016. 

TODO!

In [ ]:
#| default_exp RONA

In [ ]:
#| hide
from fastcore.utils import *
from nbdev.showdoc import *
import numpy as np

In [ ]:
#| export
class RONA:
    "Risk of non-adaptedness genomic offset statistic."
    def __init__(self): 
        self.B = None
    def __str__(self):
        if self.B is None: 
            return "RONA model. Please, fit me first using the `fit()` method."
        else: 
            p, l = self.B.shape
            return f"RONA model. Fitted with {p-1} environmental covariates and {l} alleles."
    __repr__ = __str__

In order to use the model we have first to initialize it.  

In [ ]:
model = RONA()

Then, we have to fit the model (that is, find the the least squares solutions) to

$$
\mathbf y \approx \mathbf x \mathbf b + \mathbf w
$$

where $\mathbf y = [y_1, \dots, y_L], y_l \in \mathbb [0, 1]$ is a vector with the encoded allele frequencies and $\mathbf x = [x_1, \dots, x_P], x_p \in \mathbb R$ is a vector with the environmental covariates. TODO: add details. 

In [ ]:
#|export
@patch
def fit(self:RONA,
        Y: np.ndarray, # Allele frequency matrix (nxL)
        X: np.ndarray): # Environmental matrix (nxP)
    "Fits the RONA model. "
    n1, L = Y.shape
    n2, P = X.shape
    if n1 != n2: 
        raise ValueError("Dimensions of array don't match")
    X = np.hstack((np.ones((X.shape[0], 1)), X)) 
    B = np.linalg.lstsq(X, Y)[0]
    self.B = B

The `fit()` method expects as input an allele matrix $\mathbf Y$ and an environmental matrix $\mathbf X$ with as many rows as individuals. For now, let us simulate them under the ¿generative model?: 

In [ ]:
import numpy as np
N, L, P = 100, 1_000, 3
rng = np.random.default_rng(1000)
X = rng.normal(loc=0.0, scale=1.0, size=(N, P))
p = rng.uniform(low=0, high=1, size = (1, L))
B = rng.normal(loc=0.0, scale=0.1, size = (P, L))
Y = X@B + np.ones((N, 1))@p
Y = Y.clip(0, 1)
assert X.shape == (N, P)
assert Y.shape == (N, L)

In [ ]:
indices = rng.permutation(N)
training_idx, test_idx = indices[:60], indices[60:]
Xtrain, Xpredict = X[training_idx,:], X[test_idx,:]
Ytrain, Ypredict = Y[training_idx,:], Y[test_idx,:]
model.fit(Ytrain, Xtrain)

Now, we can make predictions: 

In [ ]:
#| export 
@patch
def predict(self:RONA,
        X: np.ndarray # Environmental matrix (nxP)
           )-> np.ndarray: # Predicted allele frequencies
    "Predicts the allele frequencies for a given environmental matrix. "    
    B = self.B
    if X.shape[1] != (B.shape[0]-1):
        raise ValueError("Dimensions of array don't match")
    X = np.hstack((np.ones((X.shape[0], 1)), X)) 
    return X@B

In [ ]:
mse = (np.square(model.predict(Xpredict) - Ypredict)).mean()
mse

np.float64(0.001041810551801656)

Finally, we can predict the genomic offset under two different environments: 

In [ ]:
#| export 
@patch
def genomic_offset(self:RONA,
        X: np.ndarray, # Environmental matrix (nxP)
        Xstar: np.ndarray, # Altered environmental matrix (nxP)
           )-> np.ndarray: # A vector of genomic offsets (n)
    "Predicts the allele frequencies for a given environmental matrix. "    
    B = self.B
    if X.shape != Xstar.shape:
        raise ValueError("Dimensions of array don't match")
    X = np.hstack((np.ones((X.shape[0], 1)), X)) 
    Xstar = np.hstack((np.ones((Xstar.shape[0], 1)), Xstar)) 
    return np.sum(np.abs(X @ self.B - Xstar @ self.B), axis=1) / model.B.shape[1]

As expected, the genomic offset is zero if both environmental matrixes are identical: 

In [ ]:
model.genomic_offset(Xpredict, Xpredict)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0.])

In [ ]:
offset = model.genomic_offset(Xpredict, Xpredict+rng.normal(size=Xpredict.shape))
offset

array([0.13751402, 0.09202521, 0.0600742 , 0.12013233, 0.06577858,
       0.14360362, 0.04944554, 0.12189046, 0.06529109, 0.14930976,
       0.13746642, 0.12096128, 0.09761957, 0.23370016, 0.08638076,
       0.11536754, 0.10702193, 0.12890098, 0.1214036 , 0.01850179,
       0.13102646, 0.07015995, 0.07971816, 0.05795829, 0.17571149,
       0.09132069, 0.13188631, 0.03935012, 0.13387128, 0.17142883,
       0.08332022, 0.05644843, 0.07243936, 0.12327688, 0.09299116,
       0.06894027, 0.10736288, 0.06372873, 0.05408654, 0.15056631])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()